# Materials Project Element Distribution by Arity


In [ ]:
# mp-api: Materials Project data
%pip install pymatviz mp-api

In [ ]:
import ipywidgets
import pandas as pd
from mp_api.client import MPRester

import pymatviz as pmv
from pymatviz.enums import Key


__author__ = "Janosh Riebesell"
__date__ = "2022-07-21"

In [ ]:
print(", ".join(MPRester().materials.summary.available_fields))

builder_meta, nsites, elements, nelements, composition, composition_reduced, formula_pretty, formula_anonymous, chemsys, volume, density, density_atomic, symmetry, property_name, material_id, deprecated, deprecation_reasons, last_updated, origins, warnings, structure, task_ids, uncorrected_energy_per_atom, energy_per_atom, formation_energy_per_atom, energy_above_hull, is_stable, equilibrium_reaction_energy_per_atom, decomposes_to, xas, grain_boundaries, band_gap, cbm, vbm, efermi, is_gap_direct, is_metal, es_source_calc_id, bandstructure, dos, dos_energy_up, dos_energy_down, is_magnetic, ordering, total_magnetization, total_magnetization_normalized_vol, total_magnetization_normalized_formula_units, num_magnetic_sites, num_unique_magnetic_sites, types_of_magnetic_species, k_voigt, k_reuss, k_vrh, g_voigt, g_reuss, g_vrh, universal_anisotropy, homogeneous_poisson, e_total, e_ionic, e_electronic, n, e_ij_max, weighted_surface_energy_EV_PER_ANG2, weighted_surface_energy, weighted_work_func

In [ ]:
# MPRester reads your API key from the MP_API_KEY environment variable
with MPRester(use_document_model=False) as mpr:
    mp_data = mpr.materials.summary.search(
        # nelements=[4, None], # 4 or less elements
        fields=[Key.mat_id, "formula_pretty", "nelements"]
    )

In [ ]:
df_mp = pd.DataFrame(map(dict, mp_data)).set_index("material_id")  # ty: ignore[invalid-argument-type]
df_mp.head()

In [ ]:
# uncomment line to cache large MP data
# %store df_mp

# uncomment line to load cached MP data from disk
# %store -r df_mp

Stored 'df_mp' (DataFrame)


In [ ]:
elem_counts_by_arity = {
    label: pmv.count_elements(df_mp.query(f"nelements == {idx}")["formula_pretty"])
    for idx, label in enumerate(["unary", "binary", "ternary", "quaternary"], 1)
}

compound_counts_by_arity = {
    key: (df_mp.nelements == idx).sum()
    for idx, key in enumerate(elem_counts_by_arity, 1)
}

In [ ]:
for arity_label, elem_counts in elem_counts_by_arity.items():
    n_compounds = compound_counts_by_arity[arity_label]
    title = f"Element distribution of {n_compounds:,} {arity_label} compounds in MP"
    pmv.ptable_heatmap(elem_counts, log=True, color_bar={"title": title}).show()

In [ ]:
# Pick an arity from the dropdown to update the heatmap in place
widget = pmv.ptable_heatmap(elem_counts_by_arity["unary"], heat_mode="percent")
dropdown = ipywidgets.Dropdown(options=list(elem_counts_by_arity), description="Arity")


def show_arity(change: dict) -> None:
    """Swap in the element percentages of the selected arity."""
    counts = elem_counts_by_arity[change["new"]]
    new_widget = pmv.ptable_heatmap(counts, heat_mode="percent")
    widget.heatmap_values = new_widget.heatmap_values


dropdown.observe(show_arity, names="value")
ipywidgets.VBox([dropdown, widget])